## CNN Model
## Overview
This notebook designs and justifies a CNN architecture for adversarial robustness evaluation:
- Model architecture design
- Why this specific architecture?
- How it aligns with CIFAR-10 and robustness evaluation
- Complete model summary
## Research Context
For evaluating CNN robustness to adversarial perturbations, we need:
- Feature extraction capabilities (Conv layers)
- Non-linearity (ReLU activations)  
- Regularization (Dropout)
- Interpretable outputs (10-class softmax)


In [1]:

#Imports
import os
import numpy as np
import pickle
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print("Libraries imported")

# Load preprocessed data
with open('data/processed/cifar10_preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)
    X_train = data['X_train']
    y_train = data['y_train']
    X_val = data['X_val']
    y_val = data['y_val']
    class_names = data['class_names']

print(f"  Preprocessed data loaded")
print(f"  Training: {X_train.shape}")
print(f"  Validation: {X_val.shape}")

Libraries imported
  Preprocessed data loaded
  Training: (40000, 32, 32, 3)
  Validation: (10000, 32, 32, 3)



## CNN Architecture Design & Justification
### Why Convolutional Neural Networks?
**Advantages for CIFAR-10:**
- **Local Connectivity:** Conv filters capture local spatial features (edges, textures)
- **Parameter Sharing:** Weights are reused across image, reducing parameters
- **Translation Invariance:** Learns features regardless of position in image
- **Hierarchical Features:** Early layers learn edges → mid layers learn shapes → late layers learn objects
### Architecture Components
#### **Input Layer**
- Shape: (32, 32, 3) - CIFAR-10 RGB images
- Pixel values normalized to [0, 1]
#### **Convolutional Block 1**
- 32 filters, 3×3 kernel (small receptive field)
- ReLU activation (non-linearity)
- MaxPool 2×2 (spatial dimensionality reduction)
- Dropout 0.25 (regularization against overfitting)
#### **Convolutional Block 2**
- 64 filters, 3×3 kernel (larger feature maps)
- ReLU activation
- MaxPool 2×2
- Dropout 0.25
#### **Convolutional Block 3**
- 128 filters, 3×3 kernel (deeper features)
- ReLU activation
- Dropout 0.25 (no pool - preserves spatial info before dense)
#### **Dense Layers**
- Flatten conv output
- Dense 256 (feature extraction)
- Dropout 0.5 (stronger regularization on dense)
- Dense 10 + Softmax (10-class output probabilities)
### Design Rationale
**Why Progressive Filter Growth (32→64→128)?**
- Allows model to learn increasingly abstract features
- Standard practice in CNN design for image classification
**Why Dropout?**
- Prevents overfitting by randomly deactivating neurons
- Crucial for generalization to adversarial examples
- Higher dropout on dense (0.5) vs conv (0.25) layers
**Why MaxPooling?**
- Reduces spatial dimensions, improving efficiency
- Adds translation invariance (robust to small shifts)
- Helps prevent overfitting
**Why this architecture for robustness?**
- Moderate complexity: Not too simple (underfits), not too complex (overfits)
- Regularization (Dropout) helps learn robust features
- Will help us understand CNN limitations vs adversarial attacks

In [2]:

## Building the CNN Model


def create_cnn_model(input_shape=(32, 32, 3), num_classes=10):
    """
    Create CNN model for CIFAR-10 classification and adversarial robustness evaluation.
    
    Args:
        input_shape: Tuple of input dimensions (H, W, C)
        num_classes: Number of output classes
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # Input layer (implicit)
        layers.Input(shape=input_shape),
        
        # Block 1: Conv → MaxPool → Dropout
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1'),
        layers.MaxPooling2D((2, 2), name='pool1'),
        layers.Dropout(0.25, name='dropout1'),
        
        # Block 2: Conv → MaxPool → Dropout
        layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2'),
        layers.MaxPooling2D((2, 2), name='pool2'),
        layers.Dropout(0.25, name='dropout2'),
        
        # Block 3: Conv → Dropout (no pooling to preserve spatial info)
        layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3'),
        layers.Dropout(0.25, name='dropout3'),
        
        # Dense layers
        layers.Flatten(name='flatten'),
        layers.Dense(256, activation='relu', name='dense1'),
        layers.Dropout(0.5, name='dropout4'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ])
    
    return model

# Create model
model = create_cnn_model()

# Compile with appropriate optimizer and loss
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("CNN Model created successfully")

CNN Model created successfully


In [3]:

## Model Summary & Analysis


print("\n" + "="*40)
print("CNN MODEL ARCHITECTURE")
print("="*40)

model.summary()

print("\n" + "="*40)
print("MODEL STATISTICS")
print("="*40)

total_params = model.count_params()
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {sum([np.prod(w.shape) for w in model.trainable_weights]):,}")

# Calculate receptive field
print("\n" + "-"*40)
print("Effective Receptive Field:")
print("-"*40)
print("Conv1 (3×3): Receptive field = 3×3")
print("Conv2 (3×3, after Pool): Effective field = 3×3 (at 4× stride)")
print("Conv3 (3×3): Further expansion")
print("→ Model can capture 'global' features from local patches")

print("\nModel ready for training")


CNN MODEL ARCHITECTURE


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout3 (Dropout)              │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout4 (Dropout)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,193,226 (8.37 MB)

 Trainable params: 2,193,226 (8.37 MB)

 Non-trainable params: 0 (0.00 B)


MODEL STATISTICS
Total Parameters: 2,193,226
Trainable Parameters: 2,193,226

----------------------------------------
Effective Receptive Field:
----------------------------------------
Conv1 (3×3): Receptive field = 3×3
Conv2 (3×3, after Pool): Effective field = 3×3 (at 4× stride)
Conv3 (3×3): Further expansion
→ Model can capture 'global' features from local patches

Model ready for training


In [4]:

## Save Model for Training

os.makedirs('models', exist_ok=True)

# Use .keras format (modern, recommended)
model.save('models/cnn_architecture.keras')

print(" Model saved to 'models/cnn_architecture.keras'")


 Model saved to 'models/cnn_architecture.keras'
